# Automatic Speech Recognition -- notebook for kaggle submission

## Конфигурация

In [ ]:
import os

REPO_DIR = '/kaggle/working/repo'

GITHUB_RELEASE_WEIGHTS_URL = 'https://github.com/sadevans/ASR_numbers_recognition_rus/blob/main/checkpoints/best.pt'

KAGGLE_WEIGHTS_PATH = None  # например '/kaggle/input/gp1-weights/best.pt'

WORKING = '/kaggle/working'


def find_competition_root() -> str:
    candidates = [
        '/kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge',
        '/kaggle/input/asr-2026-spoken-numbers-recognition-challenge',
    ]
    for c in candidates:
        if os.path.isfile(os.path.join(c, 'test.csv')):
            return c
    raise FileNotFoundError(
        'Не найден test.csv. Добавьте Competition Data в ноутбук.'
    )


COMP_ROOT = find_competition_root()
TEST_CSV = os.path.join(COMP_ROOT, 'test.csv')
DEV_CSV = os.path.join(COMP_ROOT, 'dev.csv')

print('COMP_ROOT =', COMP_ROOT)

COMP_ROOT = /kaggle/input/competitions/asr-2026-spoken-numbers-recognition-challenge


## Зависимости, `git clone` кода, загрузка весов с GitHub Release

In [ ]:
!pip install -q soundfile num2words

In [ ]:
! git clone https://github.com/sadevans/ASR_numbers_recognition_rus.git


Cloning into 'Speech_ITMO'...
remote: Enumerating objects: 551, done.
remote: Counting objects: 100% (61/61), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 551 (delta 15), reused 57 (delta 11), pack-reused 490 (from 2)
Receiving objects: 100% (551/551), 111.31 MiB | 10.90 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [ ]:
%cd /kaggle/working/ASR_numbers_recognition_rus/

[Errno 2] No such file or directory: 'gp1'
/kaggle/working


In [ ]:
CHECKPOINT_PATH = '/kaggle/working/Speech_ITMO/gp1/checkpoints/best.pt'

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 3.1 MB/s eta 0:00:00


In [5]:
import sys

CODE_SRC = '/kaggle/working/Speech_ITMO/gp1/src'


import torch
import torchaudio
import pandas as pd

from src.model import DigitCTCModel
from src.char_vocab import greedy_ctc_decode
from src.text_normalize import denormalize_digits_for_submission, normalize_transcription
from src.metrics import cer_over_utterances

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('CODE_SRC =', CODE_SRC)
print('CHECKPOINT_PATH =', CHECKPOINT_PATH)
print('device:', device)

ModuleNotFoundError: No module named 'src'

## Загрузка весов и сборка модели

Чекпоинт уже лежит по `CHECKPOINT_PATH` (скачан с **GitHub Release** или из Kaggle Dataset). Гиперпараметры архитектуры читаем из `checkpoint['args']` (в т.ч. SpecAugment).

In [ ]:
try:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
except TypeError:
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device)
train_args = ckpt.get('args') or {}
use_spec_augment = not bool(train_args.get('no_spec_augment', False))

model = DigitCTCModel(use_spec_augment=use_spec_augment).to(device)
model.load_state_dict(ckpt['model_state_dict'], strict=True)
model.eval()

print('Loaded epoch', ckpt.get('epoch'), 'val_cer_mean', ckpt.get('val_cer_mean'))

## Func

In [ ]:
def load_mono_16k(path):
    wav, sr = torchaudio.load(path)
    if wav.dim() == 2 and wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)
    if sr != 16_000:
        wav = torchaudio.functional.resample(wav, sr, 16_000)
    return wav


@torch.no_grad()
def predict_digit_strings(model, paths, device):
    waves = [load_mono_16k(p) for p in paths]
    lengths = torch.tensor([w.shape[-1] for w in waves], dtype=torch.long)
    max_len = int(lengths.max())
    padded = torch.stack(
        [torch.nn.functional.pad(w, (0, max_len - w.shape[-1])) for w in waves]
    )
    log_probs, _ = model(padded.to(device), lengths.to(device))
    t_steps, batch_size, _ = log_probs.shape
    out: list[str] = []
    for b in range(batch_size):
        idx = log_probs[:t_steps, b, :].argmax(dim=-1).detach().cpu().tolist()
        out.append(greedy_ctc_decode(idx))
    return out


def predict_submission_integers(model, paths, device):
    digit_strs = predict_digit_strings(model, paths, device)
    return [denormalize_digits_for_submission(s) for s in digit_strs]

## Опционально: проверяемся на валидации еще разок

In [ ]:
DEV_CSV

In [ ]:
RUN_DEV_SANITY = os.path.isfile(DEV_CSV)

if RUN_DEV_SANITY:
    dev_df = pd.read_csv(DEV_CSV)
    batch_size = 16
    refs: list[str] = []
    hyps: list[str] = []
    for start in range(0, len(dev_df), batch_size):
        print(start)
        chunk = dev_df.iloc[start : start + batch_size]
        paths = [os.path.join(COMP_ROOT, fn) for fn in chunk['filename']]
        digit_hyps = predict_digit_strings(model, paths, device)
        for raw_label, hyp_digits in zip(chunk['transcription'], digit_hyps):
            _, ref_digits = normalize_transcription(str(raw_label), 'digits')
            refs.append(ref_digits)
            hyps.append(hyp_digits)
    dev_cer = cer_over_utterances(refs, hyps)
    print(f'dev CER (digit strings, no submission denorm): {dev_cer:.4f}')
else:
    print('dev.csv не найден — пропуск sanity-check.')

## Test: инференс и формирование `submission.csv`

In [ ]:
TEST_CSV

In [ ]:
test_df = pd.read_csv(TEST_CSV)
batch_size = 16
pred_ints: list[int] = []

for start in range(0, len(test_df), batch_size):
    chunk = test_df.iloc[start : start + batch_size]
    paths = [os.path.join(COMP_ROOT, fn) for fn in chunk['filename']]
    pred_ints.extend(predict_submission_integers(model, paths, device))

test_df = test_df.copy()
test_df['transcription'] = pred_ints
out_path = os.path.join(WORKING, 'submission.csv')
test_df[['filename', 'transcription']].to_csv(out_path, index=False)
print('Saved', out_path)
test_df.head()

In [ ]:
out_path